<a href="https://colab.research.google.com/github/dmecoyfin/dmeyf2026/blob/main/ensembles/z494_TareaHogar_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# COMPETENCIA 1





#### 2.1  Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [ ]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Mounted at /content/.drive


Para correr la siguiente celda es fundamental en Arranque en Frio haber copiado el archivo kaggle.json al Google Drive, en la carpeta indicada en el instructivo

<br>los siguientes comando estan en shell script de Linux
*   Crear las carpetas en el Google Drive
*   "instalar" el archivo kaggle.json desde el Google Drive a la virtual machine para que pueda ser utilizado por la libreria  kaggle de Python
*   Bajar el  **dataset_pequeno**  al  Google Drive  y tambien al disco local de la virtual machine que esta corriendo Google Colab



In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/dmeyf"
mkdir -p "/content/buckets"
ln -s "/content/.drive/My Drive/dmeyf" /content/buckets/b1


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets



archivo_origen="https://storage.googleapis.com/open-courses/dmeyf2026-9c6f/competencia_01_crudo.csv"
archivo_destino="/content/datasets/competencia_01_crudo.csv"
archivo_destino_bucket="/content/buckets/b1/datasets/competencia_01_crudo.csv"

if ! test -f $archivo_destino_bucket; then
  wget  $archivo_origen  -O $archivo_destino_bucket
fi


if ! test -f $archivo_destino; then
  cp  $archivo_destino_bucket  $archivo_destino
fi


## Generacion de la clase_ternaria

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Tipe -> Runtime type -> R

In [ ]:
require( "data.table" )

# leo el dataset
dataset <- fread("/content/datasets/competencia_01_crudo.csv" )

# La variable rota se anula UNICAMENTE en mayo de 2021.
# Se corrige antes de crear variables temporales.
dataset[foto_mes == 202105, ccajas_depositos := NA_real_]

# calculo el periodo0 consecutivo
dsimple <- dataset[, list(
    "pos" = .I,
    numero_de_cliente,
    periodo0 = as.integer(foto_mes/100)*12 +  foto_mes%%100 ) ]


# ordeno
setorder( dsimple, numero_de_cliente, periodo0 )

# calculo topes
periodo_ultimo <- dsimple[, max(periodo0) ]
periodo_anteultimo <- periodo_ultimo - 1


# calculo los leads de orden 1 y 2
dsimple[, c("periodo1", "periodo2") :=
    shift(periodo0, n=1:2, fill=NA, type="lead"),  numero_de_cliente ]

# assign most common class values = "CONTINUA"
dsimple[ periodo0 < periodo_anteultimo, clase_ternaria := "CONTINUA" ]

# calculo BAJA+1
dsimple[ periodo0 < periodo_ultimo &
    ( is.na(periodo1) | periodo0 + 1 < periodo1 ),
    clase_ternaria := "BAJA+1" ]

# calculo BAJA+2
dsimple[ periodo0 < periodo_anteultimo & (periodo0+1 == periodo1 )
    & ( is.na(periodo2) | periodo0 + 2 < periodo2 ),
    clase_ternaria := "BAJA+2" ]


# pego el resultado en el dataset original y grabo
setorder( dsimple, pos )
dataset[, clase_ternaria := dsimple$clase_ternaria ]

fwrite( dataset,
    file =  "/content/datasets/competencia_01.csv.gz",
    sep = ","
)

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%




In [ ]:
setorder( dataset, foto_mes, clase_ternaria, numero_de_cliente)
dataset[, .N, list(foto_mes, clase_ternaria)]

foto_mes,clase_ternaria,N
<int>,<chr>,<int>
202103,BAJA+1,1019
202103,BAJA+2,960
202103,CONTINUA,160921
202104,BAJA+1,964
202104,BAJA+2,1139
202104,CONTINUA,161181
202105,BAJA+1,1143
202105,BAJA+2,870
202105,CONTINUA,161755


### 2.2 Optimizacion Hiperparámetros

Esta parte se debe correr con el runtime en lenguaje R Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

### 2.2.1 Inicio

limpio el ambiente de R

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Sun Oct 11 05:33:52 PM 2026"

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,749054,40.1,1473300,78.7,1473300,78.7
Vcells,1409282,10.8,170859802,1303.6,212616006,1622.2


### 2.2.2 Carga de Librerias

Esta parte lleva varios minutos la primera vez en Google Colab

In [ ]:
# cargo las librerias que necesito
require("data.table")
require("parallel")

if(!require("R.utils")) install.packages("R.utils")
require("R.utils")

if( !require("primes") ) install.packages("primes")
require("primes")

if( !require("utils") ) install.packages("utils")
require("utils")

if( !require("rlist") ) install.packages("rlist")
require("rlist")

if( !require("yaml")) install.packages("yaml")
require("yaml")

if( !require("lightgbm") ) install.packages("lightgbm")
require("lightgbm")

if( !require("DiceKriging") ) install.packages("DiceKriging")
require("DiceKriging")

if( !require("mlrMBO") ) install.packages("mlrMBO")
require("mlrMBO")

Loading required package: parallel

Loading required package: R.utils

Warning message in library(package, lib.loc = lib.loc, character.only = TRUE, logical.return = TRUE, :
“there is no package called ‘R.utils’”
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘R.oo’, ‘R.methodsS3’


Loading required package: R.utils

Loading required package: R.oo

Loading required package: R.methodsS3

R.methodsS3 v1.8.2 (2022-06-13 22:00:14 UTC) successfully loaded. See ?R.methodsS3 for help.

R.oo v1.27.1 (2025-05-02 21:00:05 UTC) successfully loaded. See ?R.oo for help.


Attaching package: ‘R.oo’


The following object is masked from ‘package:R.methodsS3’:

    throw


The following objects are masked from ‘package:methods’:

    getClasses, getMethods


The following objects are masked from ‘package:base’:

    attach, detach, load, save


R.utils v2.13.0 (2025-02-24 21:20:02 UTC) successfully loaded. See ?R.utils for help.


At

### 2.2.3 Definicion de Parametros

aqui debe cargar SU semilla primigenia
<br>recuerde cambiar el numero de experimento en cada corrida nueva

In [ ]:
PARAM <- list()
PARAM$experimento <- "Entrega1Prueba1"
PARAM$semilla_primigenia <- 170383


In [ ]:
# Optimización: validación temporal con junio; el entrenamiento final incluye los cuatro meses.
PARAM$train <- c(202103, 202104)
PARAM$validacion <- 202106
PARAM$train_final <- c(202103, 202104, 202105, 202106)
PARAM$future <- 202108
PARAM$semilla_kaggle <- 314159
PARAM$cortes <- c(10000L, 10500L, 11000L, 11500L, 12000L)
PARAM$n_modelos <- 10L
PARAM$top_configuraciones <- 10L

In [ ]:
# Sin undersampling en la primera competencia, incluida la BO.
PARAM$trainingstrategy$undersampling <- 1.0

In [ ]:
# Parametros LightGBM
# parametros fijos del LightGBM que se pisaran con la parte variable de la BO
PARAM$lgbm$param_fijos <-  list(
  boosting= "gbdt", # puede ir  dart  , ni pruebe random_forest
  objective= "binary",
  metric= "auc",
  first_metric_only= TRUE,
  boost_from_average= TRUE,
  feature_pre_filter= FALSE,
  force_row_wise= TRUE, # para reducir warnings
  verbosity= -100,

  seed= PARAM$semilla_primigenia,

  max_depth= -1L, # -1 significa no limitar,  por ahora lo dejo fijo
  min_gain_to_split= 0, # min_gain_to_split >= 0
  lambda_l1= 0.0, # lambda_l1 >= 0.0
  lambda_l2= 0.0, # lambda_l2 >= 0.0
  max_bin= 31L, # lo debo dejar fijo, no participa de la BO

  bagging_fraction= 1.0, # 0.0 < bagging_fraction <= 1.0
  pos_bagging_fraction= 1.0, # 0.0 < pos_bagging_fraction <= 1.0
  neg_bagging_fraction= 1.0, # 0.0 < neg_bagging_fraction <= 1.0
  is_unbalance= FALSE, #
  scale_pos_weight= 1.0, # scale_pos_weight > 0.0

  drop_rate= 0.1, # 0.0 < neg_bagging_fraction <= 1.0
  max_drop= 50, # <=0 means no limit
  skip_drop= 0.5, # 0.0 <= skip_drop <= 1.0

  extra_trees= FALSE,

  num_iterations= 1200,
  learning_rate= 0.005,
  feature_fraction= 0.50,
  feature_fraction_bynode= 0.20,
  num_leaves= 750,
  min_data_in_leaf= 0
)

Aqui se definen los hiperparámetros de LightGBM que participan de la Bayesian Optimization
<br> si es un numero entero debe ir  makeIntegerParam
<br> si es un numero real (con decimales) debe ir  makeNumericParam
<br> es muy importante leer cuales son un lower y upper  permitidos y ademas razonables

In [ ]:
# Los únicos hiperparámetros optimizados (los demás permanecen fijos).
PARAM$hyperparametertuning$hs <- makeParamSet(
  makeIntegerParam("num_iterations", lower= 64L, upper= 4096L),
  makeIntegerParam("num_leaves", lower= 16L, upper= 2048L),
  makeNumericParam("min_sum_hessian_in_leaf", lower= 1e-06, upper= 0.1)
)

A mayor cantidad de hiperparámetros, se debe aumentar las iteraciones de la Bayesian Optimization
<br> 30 es un valor muy tacaño, pero corre rápido
<br> deberia partir de 50, alcanzando los 100 si se dispone de tiempo

In [ ]:
PARAM$hyperparametertuning$iteraciones <- 30L # ajustar según presupuesto computacional

In [ ]:
# particionar agrega una columna llamada fold a un dataset
#   que consiste en una particion estratificada segun agrupa
# particionar( data=dataset, division=c(70,30),
#  agrupa=clase_ternaria, seed=semilla)   crea una particion 70, 30

particionar <- function(data, division, agrupa= "", campo= "fold", start= 1, seed= NA) {
  if (!is.na(seed)) set.seed(seed, "L'Ecuyer-CMRG")

  bloque <- unlist(mapply(
    function(x, y) {rep(y, x)},division, seq(from= start, length.out= length(division))))

  data[, (campo) := sample(rep(bloque,ceiling(.N / length(bloque))))[1:.N],by= agrupa]
}

In [ ]:
# iniciliazo el dataset de realidad, para medir ganancia
realidad_inicializar <- function( pfuture, pparam) {

  # datos para verificar la ganancia
  drealidad <- pfuture[, list(numero_de_cliente, foto_mes, clase_ternaria)]

  particionar(drealidad,
    division= c(3, 7),
    agrupa= "clase_ternaria",
    seed= PARAM$semilla_kaggle
  )

  return( drealidad )
}

In [ ]:
# evaluo ganancia en los datos de la realidad

realidad_evaluar <- function( prealidad, pprediccion) {

  prealidad[ pprediccion,
    on= c("numero_de_cliente", "foto_mes"),
    predicted:= i.Predicted
  ]

  tbl <- prealidad[, list("qty"=.N), list(fold, predicted, clase_ternaria)]

  res <- list()
  res$public  <- tbl[fold==1 & predicted==1L, sum(qty*ifelse(clase_ternaria=="BAJA+2", 1072500, -27500))]/0.3
  res$private <- tbl[fold==2 & predicted==1L, sum(qty*ifelse(clase_ternaria=="BAJA+2", 1072500, -27500))]/0.7
  res$total <- tbl[predicted==1L, sum(qty*ifelse(clase_ternaria=="BAJA+2", 1072500, -27500))]

  prealidad[, predicted:=NULL]
  return( res )
}

### 2.2.4  Preprocesamiento

In [ ]:
# carpeta de trabajo

setwd("/content/buckets/b1/exp")
experimento_folder <- paste0("HT", PARAM$experimento)
dir.create(experimento_folder, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento_folder ))

In [ ]:

library(data.table)

t_total <- proc.time()[["elapsed"]]

# ============================================================
# 1. LECTURA Y CORRECCION DE MAYO
# ============================================================
dataset <- fread(
  "/content/datasets/competencia_01.csv.gz",
  stringsAsFactors = FALSE,
  showProgress = TRUE
)

dataset[foto_mes == 202105L, ccajas_depositos := NA_real_]

excluir <- c(
  "numero_de_cliente", "foto_mes", "clase_ternaria",
  "clase01", "azar", "training", "fold"
)

variables_originales <- setdiff(names(dataset), excluir)
variables_numericas <- variables_originales[
  vapply(dataset[, ..variables_originales], is.numeric, logical(1))
]

otras <- setdiff(variables_originales, variables_numericas)
if (length(otras)) {
  message("Variables no numericas sin lags: ",
          paste(otras, collapse = ", "))
}

setorder(dataset, numero_de_cliente, foto_mes)

cat("Variables numericas:", length(variables_numericas), "\n")
cat("Filas:", nrow(dataset), "\n")

# ============================================================
# 2. LAGS Y DELTA LAGS (ORDENES 1 Y 2)
# ============================================================
t0 <- proc.time()[["elapsed"]]

mes <- (dataset$foto_mes %/% 100L) * 12L +
       (dataset$foto_mes %% 100L)

cliente <- dataset$numero_de_cliente
n <- nrow(dataset)

for (orden in 1:2) {

  # Una sola mascara para TODAS las variables
  valido <- rep(FALSE, n)

  if (n > orden) {
    actual <- seq.int(orden + 1L, n)
    anterior <- actual - orden

    valido[actual] <-
      !is.na(cliente[actual]) &
      !is.na(cliente[anterior]) &
      cliente[actual] == cliente[anterior] &
      (mes[actual] - mes[anterior] == orden)
  }

  # Sin by=cliente ni agrupamientos repetidos
  for (v in variables_numericas) {

    previo <- shift(dataset[[v]], n = orden)
    previo[!valido] <- NA_real_

    set(
      dataset,
      j = paste0(v, "_lag", orden),
      value = previo
    )

    set(
      dataset,
      j = paste0(v, "_delta_lag", orden),
      value = dataset[[v]] - previo
    )
  }

  cat(
    "Orden", orden, "completado en",
    round(proc.time()[["elapsed"]] - t0, 1),
    "segundos acumulados\n"
  )
}

rm(mes, cliente, valido)
gc(verbose = FALSE)

# ============================================================
# 3. RANKINGS MENSUALES [-1, 1], CERO FIJO
# ============================================================
t_rank <- proc.time()[["elapsed"]]

ranking_cero_fijo <- function(x) {

  z <- rep(NA_real_, length(x))

  ceros <- !is.na(x) & x == 0
  z[ceros] <- 0

  neg <- which(!is.na(x) & x < 0)
  pos <- which(!is.na(x) & x > 0)

  if (length(neg)) {
    z[neg] <- -1 +
      (frank(x[neg], ties.method = "average") - 1) /
      length(neg)
  }

  if (length(pos)) {
    z[pos] <-
      frank(x[pos], ties.method = "average") /
      length(pos)
  }

  z
}

campos_ranking <- setdiff(names(dataset), excluir)
campos_ranking <- campos_ranking[
  vapply(dataset[, ..campos_ranking], is.numeric, logical(1))
]

# Calculamos los indices de cada mes UNA SOLA VEZ
indices_meses <- split(
  seq_len(nrow(dataset)),
  dataset$foto_mes
)

total <- length(campos_ranking)

for (i in seq_along(campos_ranking)) {

  v <- campos_ranking[i]
  x <- dataset[[v]]

  for (idx in indices_meses) {
    x[idx] <- ranking_cero_fijo(x[idx])
  }

  set(dataset, j = v, value = x)

  if (i %% 100L == 0L || i == total) {
    cat(
      "Rankings:", i, "/", total,
      "| Tiempo:",
      round(proc.time()[["elapsed"]] - t_rank, 1),
      "segundos\n"
    )
    flush.console()
  }
}

rm(indices_meses)
gc(verbose = FALSE)

# ============================================================
# 4. AUDITORIA
# ============================================================
stopifnot(
  all(is.na(dataset[
    foto_mes == 202105L,
    ccajas_depositos
  ]))
)

stopifnot(
  all(vapply(
    dataset[, ..campos_ranking],
    function(x) all(is.na(x) | (x >= -1 & x <= 1)),
    logical(1)
  ))
)

cat("\nPREPROCESAMIENTO TERMINADO\n")
cat("Features:", length(campos_ranking), "\n")
cat("Tiempo total:",
    round(proc.time()[["elapsed"]] - t_total, 1),
    "segundos\n")


Variables numericas: 152 
Filas: 983061 
Orden 1 completado en 4 segundos acumulados
Orden 2 completado en 7.8 segundos acumulados


,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,2468317,131.9,4374711,233.7,4374711,233.7
Vcells,717126381,5471.3,1000451139,7632.9,808570888,6169.0


Rankings: 100 / 760 | Tiempo: 12.3 segundos
Rankings: 200 / 760 | Tiempo: 21.2 segundos
Rankings: 300 / 760 | Tiempo: 27.9 segundos
Rankings: 400 / 760 | Tiempo: 35.2 segundos
Rankings: 500 / 760 | Tiempo: 43.9 segundos
Rankings: 600 / 760 | Tiempo: 50.1 segundos
Rankings: 700 / 760 | Tiempo: 56.8 segundos
Rankings: 760 / 760 | Tiempo: 61.3 segundos


,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,2469599,131.9,4374711,233.7,4374711,233.7
Vcells,757026609,5775.7,1445996926,11032.1,1445989753,11032.1



PREPROCESAMIENTO TERMINADO
Features: 760 
Tiempo total: 94.1 segundos


In [ ]:
dataset_train <- dataset[foto_mes %in% PARAM$train]
dataset_validate <- dataset[foto_mes == PARAM$validacion]

In [ ]:
# Mantener el target original: BAJA+1 Y BAJA+2 son clase positiva.
dataset_train[, clase01 := as.integer(clase_ternaria %in% c("BAJA+2", "BAJA+1"))]
dataset_validate[, clase01 := as.integer(clase_ternaria %in% c("BAJA+2", "BAJA+1"))]
stopifnot(!anyNA(dataset_train$clase01), !anyNA(dataset_validate$clase01))

In [ ]:
# Sin undersampling: se entrenan TODOS los registros de los meses seleccionados.
dataset_train[, training := 1L]

In [ ]:
# Identificadores/etiquetas fuera de las features (evita leakage).
campos_buenos <- setdiff(names(dataset_train),
  c("numero_de_cliente", "foto_mes", "clase_ternaria", "clase01", "training", "azar", "fold"))
# LightGBM recibe matriz numérica.
stopifnot(all(vapply(dataset_train[, ..campos_buenos], is.numeric, logical(1))))

In [ ]:
dtrain <- lgb.Dataset(
  data= data.matrix(dataset_train[, ..campos_buenos]),
  label= dataset_train$clase01, free_raw_data= FALSE
)
dvalidate <- lgb.Dataset(
  data= data.matrix(dataset_validate[, ..campos_buenos]),
  label= dataset_validate$clase01, reference=dtrain, free_raw_data=FALSE
)
cat("Train:", nrow(dataset_train), "Validation:", nrow(dataset_validate),
    "Features:", length(campos_buenos), "\n")

Train: 326184 Validation: 164114 Features: 760 


2.2.5 Configuracion Bayesian Optimization

In [ ]:
# BO tiene su propio historial persistente; no se sobreescribe con el resultado final.
giter <- 0L
if (file.exists("BO_evaluaciones.tsv")) {
  tb_BO <- fread("BO_evaluaciones.tsv")
  if (nrow(tb_BO)) giter <- max(tb_BO$iter, na.rm=TRUE)
}
loguear <- function(registro, archivo) {
  fila <- as.data.table(registro)
  fwrite(fila, file=archivo, sep="\t", append=file.exists(archivo),
         col.names=!file.exists(archivo))
}

In [ ]:
# Optimización AUC, validación temporal 202106.
Estimar_lightgbm <- function(xs) {
  giter <<- giter + 1L
  parametros <- modifyList(PARAM$lgbm$param_fijos, xs)
  parametros$seed <- PARAM$semilla_primigenia
  niter <- as.integer(parametros$num_iterations)
  parametros$num_iterations <- NULL
  modelo <- lgb.train(params=parametros, data=dtrain,
                       valids=list(validacion=dvalidate),
                       nrounds=niter,
                       verbose=-1)
  # AUC calculada explícitamente sobre validación temporal (evita diferencias de API).
  p <- predict(modelo, data.matrix(dataset_validate[, ..campos_buenos]))
  y <- dataset_validate$clase01
  positivos <- sum(y == 1L); negativos <- sum(y == 0L)
  stopifnot(positivos > 0, negativos > 0)
  puntaje <- (sum(rank(p, ties.method="average")[y == 1L]) -
              positivos*(positivos+1)/2) / (positivos*negativos)
  loguear(c(list(iter=giter, auc=puntaje), xs), "BO_evaluaciones.tsv")
  cat("BO", giter, "AUC", signif(puntaje, 6), "\n")
  return(puntaje)
}

In [ ]:
# Aqui comienza la configuracion de la Bayesian Optimization

# en este archivo quedan la evolucion binaria de la BO
kbayesiana <- "bayesiana.RDATA"

funcion_optimizar <- Estimar_lightgbm # la funcion que voy a maximizar

configureMlr(show.learner.output= FALSE)

# configuro la busqueda bayesiana,  los hiperparametros que se van a optimizar
# por favor, no desesperarse por lo complejo

obj.fun <- makeSingleObjectiveFunction(
  fn= funcion_optimizar, # la funcion que voy a maximizar
  minimize= FALSE, # estoy Maximizando la ganancia
  noisy= TRUE,
  par.set= PARAM$hyperparametertuning$hs, # definido al comienzo del programa
  has.simple.signature= FALSE # paso los parametros en una lista
)

# cada 600 segundos guardo el resultado intermedio
ctrl <- makeMBOControl(
  save.on.disk.at.time= 600, # se graba cada 600 segundos
  save.file.path= kbayesiana
) # se graba cada 600 segundos

# indico la cantidad de iteraciones que va a tener la Bayesian Optimization
ctrl <- setMBOControlTermination(
  ctrl,
  iters= PARAM$hyperparametertuning$iteraciones
) # cantidad de iteraciones

# defino el método estandar para la creacion de los puntos iniciales,
# los "No Inteligentes"
ctrl <- setMBOControlInfill(ctrl, crit= makeMBOInfillCritEI())

# establezco la funcion que busca el maximo
surr.km <- makeLearner(
  "regr.km",
  predict.type= "se",
  covtype= "matern3_2",
  control= list(trace= TRUE)
)

2.2.6 Corrida Bayesian Optimization

In [ ]:

# inicio la optimizacion bayesiana, retomando si ya existe
# es la celda mas lenta de todo el notebook

if (!file.exists(kbayesiana)) {
  bayesiana_salida <- mbo(obj.fun, learner= surr.km, control= ctrl)
} else {
  bayesiana_salida <- mboContinue(kbayesiana) # retomo en caso que ya exista
}

Warning message in mboContinue(kbayesiana):
“Tuning ended with term.iter. No need to continue. Simply returning stored result.”


In [ ]:

tb_bayesiana <- as.data.table(bayesiana_salida$opt.path)
colnames( tb_bayesiana)

[1] "num_iterations"          "num_leaves"             
 [3] "min_sum_hessian_in_leaf" "y"                      
 [5] "dob"                     "eol"                    
 [7] "error.message"           "exec.time"              
 [9] "ei"                      "error.model"            
[11] "train.time"              "prop.type"              
[13] "propose.time"            "se"                     
[15] "mean"

In [ ]:
# BO: ordenar por AUC y recuperar las mejores configuraciones distintas.
tb_bayesiana <- as.data.table(bayesiana_salida$opt.path)
tb_bayesiana[, iter := .I]
setorder(tb_bayesiana, -y)
fwrite(tb_bayesiana, file="BO_resultados.tsv", sep="\t")
hiperparametros <- c("num_iterations", "num_leaves", "min_sum_hessian_in_leaf")
stopifnot(all(hiperparametros %in% names(tb_bayesiana)))
PARAM$out$lgbm$mejores_hiperparametros <- as.list(tb_bayesiana[1, ..hiperparametros])
PARAM$out$lgbm$y <- tb_bayesiana[1, y]
configuraciones_ensamble <- unique(tb_bayesiana[, ..hiperparametros])
configuraciones_ensamble <- head(configuraciones_ensamble, PARAM$top_configuraciones)
fwrite(configuraciones_ensamble, "configuraciones_ensamble.tsv", sep="\t")

In [ ]:
write_yaml( PARAM, file="PARAM.yml")

In [ ]:
print(PARAM$out$lgbm$mejores_hiperparametros)
print(PARAM$out$lgbm$y)

$num_iterations
[1] 2733

$num_leaves
[1] 1273

$min_sum_hessian_in_leaf
[1] 0.01675586

[1] 0.9181958


## 2.3  Produccion

### Final Training
Construyo el modelo final, que es uno solo, no hace ningun tipo de particion < training, validation, testing>]

In [ ]:
setwd("/content/buckets/b1/exp")
experimento <- paste0("exp", PARAM$experimento)
dir.create(experimento, showWarnings= FALSE)
setwd(file.path("/content/buckets/b1/exp", experimento))

#### Final Training Dataset

Aqui esta la gran decision de en qué meses hago el Final Training
<br> debo utilizar los mejores hiperparámetros que encontré en la  optimización bayesiana

In [ ]:
# Mantener BAJA+1 y BAJA+2 como positivos; no inventar etiqueta futura.
dataset[, clase01 := fifelse(is.na(clase_ternaria), NA_integer_,
  as.integer(clase_ternaria %in% c("BAJA+1", "BAJA+2")))]

In [ ]:
dataset_train <- dataset[foto_mes %in% PARAM$train_final]
dataset_train[,.N,clase_ternaria]

clase_ternaria,N
<chr>,<int>
CONTINUA,645999
BAJA+2,4067
BAJA+1,4000


In [ ]:
dtrain_final <- lgb.Dataset(
  data=data.matrix(dataset_train[, ..campos_buenos]),
  label=dataset_train$clase01, free_raw_data=FALSE
)

#### Final Training Hyperparameters

In [ ]:
# Las configuraciones seleccionadas por AUC varían entre modelos.
param_final <- modifyList(PARAM$lgbm$param_fijos,
  PARAM$out$lgbm$mejores_hiperparametros)
param_final

$boosting
[1] "gbdt"

$objective
[1] "binary"

$metric
[1] "auc"

$first_metric_only
[1] TRUE

$boost_from_average
[1] TRUE

$feature_pre_filter
[1] FALSE

$force_row_wise
[1] TRUE

$verbosity
[1] -100

$seed
[1] 170383

$max_depth
[1] -1

$min_gain_to_split
[1] 0

$lambda_l1
[1] 0

$lambda_l2
[1] 0

$max_bin
[1] 31

$bagging_fraction
[1] 1

$pos_bagging_fraction
[1] 1

$neg_bagging_fraction
[1] 1

$is_unbalance
[1] FALSE

$scale_pos_weight
[1] 1

$drop_rate
[1] 0.1

$max_drop
[1] 50

$skip_drop
[1] 0.5

$extra_trees
[1] FALSE

$num_iterations
[1] 2733

$learning_rate
[1] 0.005

$feature_fraction
[1] 0.5

$feature_fraction_bynode
[1] 0.2

$num_leaves
[1] 1273

$min_data_in_leaf
[1] 0

$min_sum_hessian_in_leaf
[1] 0.01675586

#### Training
Genero el modelo final, siempre sobre TODOS los datos de  final_train, sin hacer ningun tipo de undersampling de la clase mayoritaria y mucho menos cross validation.

In [ ]:
# Sin corrección por undersampling, ya que NO hubo undersampling.
param_normalizado <- copy(param_final)
param_normalizado$min_data_in_leaf <- 0L

In [ ]:
# Entrenar 10 modelos con distintas semillas Y configuraciones de BO.
# El target y todas las features son idénticos en entrenamiento y scoring.
set.seed(PARAM$semilla_primigenia)
semillas_ensamble <- sample.int(100000000L, PARAM$n_modelos)
stopifnot(nrow(configuraciones_ensamble) >= 1L)
modelos_finales <- vector("list", PARAM$n_modelos)
dfuture <- dataset[foto_mes %in% PARAM$future]
stopifnot(nrow(dfuture) > 0L, !anyDuplicated(dfuture$numero_de_cliente))
matriz_futura <- data.matrix(dfuture[, ..campos_buenos])
predicciones <- matrix(NA_real_, nrow=nrow(dfuture), ncol=PARAM$n_modelos)
registro_modelos <- vector("list", PARAM$n_modelos)
for (i in seq_len(PARAM$n_modelos)) {
  cfg <- as.list(configuraciones_ensamble[((i - 1L) %% nrow(configuraciones_ensamble)) + 1L])
  p <- modifyList(PARAM$lgbm$param_fijos, cfg)
  p$seed <- as.integer(semillas_ensamble[i])
  p$min_data_in_leaf <- 0L
  niter <- as.integer(p$num_iterations)
  p$num_iterations <- NULL
  m <- lgb.train(params=p, data=dtrain_final,
                 nrounds=niter, verbose=-1)
  modelos_finales[[i]] <- m
  predicciones[, i] <- predict(m, matriz_futura)
  lgb.save(m, sprintf("modelo_%02d.txt", i))
  registro_modelos[[i]] <- as.data.table(c(list(modelo=i, seed=p$seed), cfg))
  cat(sprintf("Modelo %02d/%02d entrenado\n", i, PARAM$n_modelos))
}
fwrite(rbindlist(registro_modelos, fill=TRUE), "modelos_semillas_configuraciones.tsv", sep="\t")

In [ ]:

library(data.table)
library(lightgbm)

# =====================================================
# ENSAMBLE DE LOS 10 MODELOS GUARDADOS
# SIN REENTRENAMIENTO
# =====================================================

# Directorio donde se guardaron modelo_01.txt, etc.
directorio_modelos <- getwd()

archivos <- file.path(
  directorio_modelos,
  sprintf("modelo_%02d.txt", 1:10)
)

# Comprobar que existan los 10 modelos
stopifnot(
  all(file.exists(archivos)),
  all(file.info(archivos)$size > 0)
)

cat("Los 10 modelos fueron encontrados.\n")

# Recuperar clientes de agosto
dfuture <- dataset[foto_mes == 202108L]

stopifnot(
  nrow(dfuture) > 0L,
  !anyDuplicated(dfuture$numero_de_cliente)
)

# Exactamente las mismas features del entrenamiento
matriz_futura <- data.matrix(
  dfuture[, ..campos_buenos]
)

# Matriz de probabilidades
pred10 <- matrix(
  NA_real_,
  nrow = nrow(dfuture),
  ncol = 10L
)

for (i in 1:10) {

  cat("Prediciendo con modelo", i, "/10\n")

  modelo <- lgb.load(archivos[i])

  pred10[, i] <- predict(
    modelo,
    matriz_futura
  )

  rm(modelo)
  gc(verbose = FALSE)
}

stopifnot(all(is.finite(pred10)))

# =====================================================
# ENSAMBLE: PROMEDIO DE LAS 10 PROBABILIDADES
# =====================================================

tb_prediccion10 <- data.table(
  numero_de_cliente = dfuture$numero_de_cliente,
  prob = rowMeans(pred10)
)

# Guardar probabilidades completas
fwrite(
  tb_prediccion10,
  "probabilidades_ensamble_10.csv.gz"
)

# Guardar tambien las predicciones individuales
pred_individuales <- as.data.table(pred10)

setnames(
  pred_individuales,
  paste0("modelo_", 1:10)
)

pred_individuales[
  , numero_de_cliente := dfuture$numero_de_cliente
]

setcolorder(
  pred_individuales,
  "numero_de_cliente"
)

fwrite(
  pred_individuales,
  "probabilidades_individuales_10.csv.gz"
)

# =====================================================
# SUBMISSIONS CON DIFERENTES CORTES
# =====================================================

setorder(
  tb_prediccion10,
  -prob,
  numero_de_cliente
)

cortes <- c(
  10000L,
  10500L,
  11000L,
  11500L,
  12000L
)

for (corte in cortes) {

  stopifnot(nrow(tb_prediccion10) >= corte)

  archivo <- sprintf(
    "submit_ensamble10_%d.csv",
    corte
  )

  fwrite(
    tb_prediccion10[
      1:corte,
      .(numero_de_cliente)
    ],
    archivo,
    col.names = FALSE,
    quote = FALSE
  )

  cat(
    "Generado:", archivo,
    "| Clientes:", corte, "\n"
  )
}

cat("\nENSAMBLE DE 10 MODELOS FINALIZADO\n")


Los 10 modelos fueron encontrados.
Prediciendo con modelo 1 /10
Prediciendo con modelo 2 /10
Prediciendo con modelo 3 /10
Prediciendo con modelo 4 /10
Prediciendo con modelo 5 /10
Prediciendo con modelo 6 /10
Prediciendo con modelo 7 /10
Prediciendo con modelo 8 /10
Prediciendo con modelo 9 /10
Prediciendo con modelo 10 /10
Generado: submit_ensamble10_10000.csv | Clientes: 10000 
Generado: submit_ensamble10_10500.csv | Clientes: 10500 
Generado: submit_ensamble10_11000.csv | Clientes: 11000 
Generado: submit_ensamble10_11500.csv | Clientes: 11500 
Generado: submit_ensamble10_12000.csv | Clientes: 12000 

ENSAMBLE DE 10 MODELOS FINALIZADO


In [ ]:
write_yaml( PARAM, file="PARAM.yml")

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")